<a href="https://colab.research.google.com/github/Bassendiaye/mes_notebooks/blob/main/CNN_DeiT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
#   Fine-tuning du modèle DeiT (Vision Transformer)
#     avec Early Stopping sur la Validation Accuracy
#     et Évaluation complète sur le Test set
# ================================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTConfig
from tqdm import tqdm
import os
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


# --------------------- Configuration ---------------------
data_dir = "/content/drive/MyDrive/Dossier_de_Basse/Data_paper_split"
batch_size = 32
image_size = 224
num_epochs = 100
lr = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------- Chargement des données ---------------------
# Transformations de base
AUGMENTATIONS = [
    A.NoOp(p=1),
    A.HorizontalFlip(p=1),
    A.VerticalFlip(p=1),
    A.ElasticTransform(alpha=1, sigma=50, p=1), # Removed alpha_affine
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=1),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
    A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=1),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
    A.Defocus(radius=(3, 5), p=1),
    A.MotionBlur(blur_limit=(3, 5), p=1),
    A.GaussianBlur(blur_limit=(3, 5), p=1)
]

# Wrapper pour Albumentations afin de fonctionner avec ImageFolder
class AlbumentationsImageFolderTransform:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, img):
        # Convert PIL Image to numpy array as expected by Albumentations
        img_np = np.array(img)
        augmented = self.transform(image=img_np)
        return augmented["image"]

train_transform = AlbumentationsImageFolderTransform(A.Compose([
    A.OneOf(AUGMENTATIONS, p=1),
    A.Resize(224, 224),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
]))

val_test_transform = AlbumentationsImageFolderTransform(A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
]))

# Chargement des datasets
train_dataset = datasets.ImageFolder(root=f"{data_dir}/TrainSet", transform=train_transform)
val_dataset   = datasets.ImageFolder(root=f"{data_dir}/ValSet", transform=val_test_transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/TestSet", transform=val_test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(" Classes:", train_dataset.classes)
print(" Mapping:", train_dataset.class_to_idx)

# --------------------- Chargement du modèle DeiT ---------------------
# Get the number of classes and create label mappings
num_labels = len(train_dataset.classes)
label_to_id = train_dataset.class_to_idx
id_to_label = {v: k for k, v in label_to_id.items()}

# Load the model directly with num_labels and ignore mismatched sizes
# Removed explicit config creation to avoid potential conflicts.
model = ViTForImageClassification.from_pretrained(
    "facebook/deit-small-patch16-224",
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True
)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# --------------------- Classe Early Stopping ---------------------
class EarlyStopping:
    def __init__(self, patience=25, delta=0.001, save_path="content/drive/MyDrive/Dossier_de_Basse/Data_paper_split/best_model_cnn_deit.pth"):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False
        self.best_model_state = None
        self.save_path = save_path
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True) # Create directory if it doesn't exist

    def __call__(self, val_acc, model):
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
            torch.save(model.state_dict(), self.save_path)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"⏳ EarlyStopping: {self.counter}/{self.patience} sans amélioration.")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            torch.save(model.state_dict(), self.save_path)
            self.counter = 0

# --------------------- Fonction d'entraînement ---------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=100):
    early_stopper = EarlyStopping(patience=25, delta=0.001)

    for epoch in range(num_epochs):
        print(f"\n----- Epoch {epoch+1}/{num_epochs} -----")

        # --- Entraînement ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs.logits, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images, labels=labels)
                loss = outputs.loss
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs.logits, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # --- Early Stopping ---
        early_stopper(val_acc, model)
        if early_stopper.early_stop:
            print(f" Early stopping activé — meilleure Val Acc: {early_stopper.best_score:.4f}")
            model.load_state_dict(torch.load(early_stopper.save_path))
            break

    return model

# --------------------- Entraînement ---------------------
trained_model = train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs)

 Classes: ['AF', 'AFC', 'AFS', 'AFSC', 'FC', 'FS', 'FSC', 'NC']
 Mapping: {'AF': 0, 'AFC': 1, 'AFS': 2, 'AFSC': 3, 'FC': 4, 'FS': 5, 'FSC': 6, 'NC': 7}


Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-small-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([8, 384]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



----- Epoch 1/100 -----


Validation: 100%|██████████| 76/76 [14:59<00:00, 11.84s/it]


Train Loss: 0.3155 | Train Acc: 0.9140
Val Loss: 0.3324 | Val Acc: 0.9175

----- Epoch 2/100 -----


...

Train Loss: 0.0407 | Train Acc: 0.9879
Val Loss: 0.2997 | Val Acc: 0.9375
⏳ EarlyStopping: 23/25 sans amélioration.

----- Epoch 48/100 -----


Validation: 100%|██████████| 76/76 [00:04<00:00, 15.85it/s]


Train Loss: 0.0382 | Train Acc: 0.9879
Val Loss: 0.2301 | Val Acc: 0.9538
⏳ EarlyStopping: 24/25 sans amélioration.

----- Epoch 49/100 -----


Validation: 100%|██████████| 76/76 [00:04<00:00, 15.61it/s]


Train Loss: 0.0434 | Train Acc: 0.9881
Val Loss: 0.2548 | Val Acc: 0.9529
⏳ EarlyStopping: 25/25 sans amélioration.
 Early stopping activé — meilleure Val Acc: 0.9600


In [ ]:
# ================================================================
# 🧪 Évaluation complète sur le jeu de test
# ================================================================
print("\n================= ÉVALUATION SUR LE TEST SET =================")

# Charger le meilleur modèle sauvegardé par Early Stopping
best_model_path = "content/drive/MyDrive/Dossier_de_Basse/Data_paper_split/best_model_cnn_deit.pth"
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()

# --- Évaluation ---
all_preds, all_labels = [], []
correct, total = 0, 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs.logits, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total
print(f"\n Test Accuracy: {test_acc:.2f}%")

# --- Rapport détaillé ---
print("\n Rapport de classification :")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

# --- Matrice de confusion graphique ---
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=test_dataset.classes,
            yticklabels=test_dataset.classes)
plt.xlabel("Prédictions")
plt.ylabel("Véritables classes")
plt.title(f"Matrice de confusion — Test Accuracy: {test_acc:.2f}%")
plt.show()